In [ ]:
!source myenv/bin/activate

In [ ]:
%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!pip install -q tokenizers matplotlib seaborn accelerate opencv-python numpy pillow accelerate bitsandbytes shapely wandb dotenv

In [ ]:
# Uninstall existing conflicting packages first
%pip uninstall -y numpy transformers

# Install compatible numpy and bleeding edge transformers
%pip install "numpy<2.0"  # Transformers often has issues with Numpy 2.0+
%pip install git+https://github.com/huggingface/transformers.git
%pip install --upgrade qwen-vl-utils

In [ ]:
from huggingface_hub import hf_hub_download, login
login()

In [ ]:
import os
import json
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import torch
import cv2
import numpy as np
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from transformers import Sam3Model, Sam3Processor
from qwen_vl_utils import process_vision_info
import re
from eval import GeoNLIEvaluator
# --- 1. MODEL SETUP ---
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
SAM_MODEL_ID = "facebook/sam3"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Qwen3-VL on {device}...")
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype="auto", device_map="auto"
)
qwen_processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"Loading SAM 3 on {device}...")
sam_processor = Sam3Processor.from_pretrained(SAM_MODEL_ID)
sam_model = Sam3Model.from_pretrained(SAM_MODEL_ID).to(device)
print("Initializing GeoNLI Evaluator...")
evaluator = GeoNLIEvaluator()

In [ ]:
# --- 1. EXTRACT TARGET CLASS FROM DESCRIPTION ---
def extract_target_class(description, qwen_model, qwen_processor):
    """Use Qwen to extract the target object class from description"""
    try:
        prompt_text = f"Extract only the main object type (1-2 words) from this description: '{description}'. Return ONLY the object type, nothing else."
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_text},
                ],
            }
        ]

        text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = qwen_processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(qwen_model.device)

        generated_ids = qwen_model.generate(**inputs, max_new_tokens=10)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = qwen_processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )[0]

        target_class = output_text.strip().lower()
        words = target_class.split()[:2]
        target_class = " ".join(words)
        
        return target_class if target_class else "object"

    except Exception as e:
        print(f"[Error] Target class extraction failed: {e}")
        words = description.lower().split()
        skip_words = {'the', 'a', 'an', 'in', 'on', 'at', 'with', 'by', 'of'}
        important_words = [w for w in words if w not in skip_words][:2]
        return " ".join(important_words) if important_words else "object"


# --- 2. GET OBB FROM MASK ---
def get_obb_from_mask(mask):
    """Convert binary mask to oriented bounding box (OBB)"""
    mask_np = mask.cpu().numpy().astype(np.uint8)
    mask_np = (mask_np > 0.5).astype(np.uint8)
    
    contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None, None
    
    # Get the largest contour
    largest_contour = max(contours, key=cv2.contourArea)
    
    # Get the minimum area rectangle (OBB)
    obb = cv2.minAreaRect(largest_contour)  # Returns ((cx, cy), (w, h), angle)
    
    return obb, largest_contour


# --- 3. SAM3 FULL IMAGE SEGMENTATION ---
def segment_full_image_sam3(image_path, target_class, sam_model, sam_processor, device):
    """Use SAM3 to segment entire image and return all OBBs with metadata"""
    try:
        original_image = Image.open(image_path).convert("RGB")
        
        print(f"[SAM3] Segmenting image with text prompt: '{target_class}'")
        
        # Prepare inputs for SAM3
        inputs = sam_processor(
            images=original_image,
            text=[target_class],
            return_tensors="pt"
        ).to(device)
        
        # Get segmentation masks
        with torch.no_grad():
            outputs = sam_model(**inputs)
        
        # Post-process to get masks and scores
        img_h, img_w = original_image.size[1], original_image.size[0]
        results = sam_processor.post_process_instance_segmentation(
            outputs,
            threshold=0.3,
            mask_threshold=0.3,
            target_sizes=[(img_h, img_w)]
        )[0]
        
        masks = results.get("masks")
        scores = results.get("scores")
        
        if masks is None or len(masks) == 0:
            print("[SAM3] No masks found")
            return None
            
        # Convert masks to OBBs
        candidate_obbs = []
        sam_metadata = []
        
        for idx, (mask, score) in enumerate(zip(masks, scores)):
            obb, contour = get_obb_from_mask(mask)
            if obb is not None:
                (cx, cy), (w, h), angle = obb
                obb_list = [cx, cy, w, h, angle]
                candidate_obbs.append(obb_list)
                sam_metadata.append({
                    "mask_id": idx,
                    "obb": obb_list,
                    "confidence": float(score.cpu().numpy()),
                    "area": float(w * h)
                })
        
        print(f"[SAM3] Found {len(candidate_obbs)} candidate OBBs")
        
        return {
            "obbs": candidate_obbs,
            "metadata": sam_metadata,
            "image_size": (img_w, img_h)
        }
        
    except Exception as e:
        print(f"[Error] SAM3 segmentation failed: {e}")
        import traceback
        traceback.print_exc()
        return None


# --- 4. QWEN OBB SELECTION ---
def select_best_obb_qwen(image_path, description, candidate_obbs, qwen_model, qwen_processor):
    """Use Qwen to select the best OBB from candidates"""
    try:
        if not candidate_obbs or len(candidate_obbs) == 0:
            print("[Qwen] No candidate OBBs to select from")
            return None, -1
        
        original_image = Image.open(image_path).convert("RGB")
        
        # Create visualization with numbered rotated boxes
        vis_image = original_image.copy()
        vis_cv = cv2.cvtColor(np.array(vis_image), cv2.COLOR_RGB2BGR)
        
        for idx, obb in enumerate(candidate_obbs):
            cx, cy, w, h, angle = obb
            box_points = cv2.boxPoints(((cx, cy), (w, h), angle))
            box_points = np.array(box_points, dtype=np.int32)
            cv2.drawContours(vis_cv, [box_points], 0, (0, 0, 255), 2)
            cv2.putText(vis_cv, str(idx), (int(cx), int(cy)), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        
        vis_image = Image.fromarray(cv2.cvtColor(vis_cv, cv2.COLOR_BGR2RGB))
        
        # Create prompt for Qwen
        obb_descriptions = "\n".join([
            f"Box {idx}: center=({obb[0]:.1f}, {obb[1]:.1f}), size=({obb[2]:.1f}x{obb[3]:.1f}), angle={obb[4]:.1f}°"
            for idx, obb in enumerate(candidate_obbs)
        ])
        
        prompt_text = f"""Given this description: "{description}"
And these candidate oriented bounding boxes:
{obb_descriptions}
Which box number best matches the description? Reply with ONLY the box number (e.g., "0" or "1" or "2")."""
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": vis_image},
                    {"type": "text", "text": prompt_text},
                ],
            }
        ]

        text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = qwen_processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(qwen_model.device)

        generated_ids = qwen_model.generate(**inputs, max_new_tokens=20)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = qwen_processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )[0]

        print(f"[Qwen] Raw selection output: '{output_text}'")
        
        # Parse the selected box number
        matches = re.findall(r'\d+', output_text)
        if matches:
            selected_idx = int(matches[0])
            if 0 <= selected_idx < len(candidate_obbs):
                print(f"[Qwen] Selected box {selected_idx}")
                return candidate_obbs[selected_idx], selected_idx
        
        # Fallback: select box with largest area
        print("[Qwen] Could not parse selection, using largest box")
        areas = [obb[2] * obb[3] for obb in candidate_obbs]
        selected_idx = int(np.argmax(areas))
        return candidate_obbs[selected_idx], selected_idx
        
    except Exception as e:
        print(f"[Error] Qwen OBB selection failed: {e}")
        import traceback
        traceback.print_exc()
        return None, -1


# --- 5. VISUALIZATION ---
def visualize_result(image_path, pred_obb, gt_obb, prompt, output_path, iou, sam_metadata, selected_idx):
    """Visualize prediction, ground truth, and all SAM candidate OBBs"""
    try:
        image = Image.open(image_path).convert("RGB")
        image_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
        
        # Draw all SAM candidate OBBs (yellow/orange)
        for meta in sam_metadata:
            obb = meta["obb"]
            cx, cy, w, h, angle = obb
            box_points = cv2.boxPoints(((cx, cy), (w, h), angle))
            box_points = np.array(box_points, dtype=np.int32)
            
            color = (0, 255, 255) if meta["mask_id"] != selected_idx else (0, 165, 255)  # Orange for selected
            thickness = 1 if meta["mask_id"] != selected_idx else 2
            cv2.drawContours(image_cv, [box_points], 0, color, thickness)
            cv2.putText(image_cv, f"{meta['mask_id']}", (int(cx), int(cy) - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        
        # Draw ground truth OBB (red)
        if gt_obb:
            gt_points = cv2.boxPoints(((gt_obb[0], gt_obb[1]), (gt_obb[2], gt_obb[3]), gt_obb[4]))
            gt_points = np.array(gt_points, dtype=np.int32)
            cv2.drawContours(image_cv, [gt_points], 0, (0, 0, 255), 3)
            cv2.putText(image_cv, "GT", tuple(map(int, gt_points[0])),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        
        # Draw predicted OBB (green)
        if pred_obb:
            pred_points = cv2.boxPoints(((pred_obb[0], pred_obb[1]), (pred_obb[2], pred_obb[3]), pred_obb[4]))
            pred_points = np.array(pred_points, dtype=np.int32)
            cv2.drawContours(image_cv, [pred_points], 0, (0, 255, 0), 3)
            cv2.putText(image_cv, "PRED", tuple(map(int, pred_points[0])),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        # Add text information
        cv2.putText(image_cv, f"IoU: {iou:.3f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        prompt_short = prompt[:50] + "..." if len(prompt) > 50 else prompt
        cv2.putText(image_cv, f"Query: {prompt_short}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        cv2.putText(image_cv, f"Selected: Box {selected_idx}", (10, 85),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 1)
        
        image_pil = Image.fromarray(cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB))
        image_pil.save(output_path)
        print(f"[Visualized] {output_path.name} - IoU: {iou:.3f}, Selected: {selected_idx}")
        
    except Exception as e:
        print(f"[Error] Visualization failed: {e}")


# --- 6. MAIN PIPELINE ---
def process_vrsbench_sam3_first(images_dir, annotations_dir, output_dir, 
                                  qwen_model, qwen_processor, sam_model, sam_processor, 
                                  evaluator, device, max_images=500):
    """Process VRSBench dataset with SAM3-first pipeline"""
    
    img_path_obj = Path(images_dir)
    ann_path_obj = Path(annotations_dir)
    out_path_obj = Path(output_dir)
    out_path_obj.mkdir(parents=True, exist_ok=True)
    
    pass_dir = out_path_obj / "pass"
    failed_dir = out_path_obj / "failed"
    pass_dir.mkdir(exist_ok=True)
    failed_dir.mkdir(exist_ok=True)

    json_files = sorted(list(ann_path_obj.glob("*.json")))[:max_images]
    print(f"Processing {len(json_files)} annotation files...")

    results = []
    correct_at_03 = 0
    correct_at_05 = 0
    total_predictions = 0
    total_horizontal_gt = 0
    total_non_horizontal_gt = 0
    image_counter = 0

    for json_file in json_files:
        with open(json_file, 'r') as f:
            data = json.load(f)

        image_name = data.get("image")
        if not image_name:
            continue
            
        image_path = img_path_obj / image_name
        if not image_path.exists():
            continue

        img = Image.open(image_path)
        img_w, img_h = img.size
        evaluator.image_width = img_w
        evaluator.image_height = img_h

        objects = data.get("objects", [])
        for obj in objects:
            desc = obj.get("referring_sentence")
            obj_id = obj.get("obj_id", 0)
            
            gt_corners = obj.get("obj_corner")
            if not desc or not gt_corners:
                continue
            
            # Convert GT corners to OBB
            gt_points = []
            for i in range(0, 8, 2):
                x = gt_corners[i] * img_w
                y = gt_corners[i+1] * img_h
                gt_points.append([x, y])
            
            gt_points = np.array(gt_points, dtype=np.float32)
            gt_obb_tuple = cv2.minAreaRect(gt_points)
            (cx, cy), (w, h), angle = gt_obb_tuple
            gt_obb = [cx, cy, w, h, angle]
            
            # Check if GT is horizontal
            angle_norm = abs(angle) % 90
            is_horizontal = angle_norm < 5 or angle_norm > 85
            iou_threshold = 0.3 if is_horizontal else 0.5
            
            image_counter += 1
            if image_counter % 50 == 0:
                print(f"\nProcessing: {image_name} - Object {obj_id}")
                print(f"Query: {desc}")
                print(f"GT is {'horizontal' if is_horizontal else 'oriented'} (IoU threshold: {iou_threshold})")
            
            # STEP 1: Extract target class
            target_class = extract_target_class(desc, qwen_model, qwen_processor)
            if image_counter % 50 == 0:
                print(f"Target class: '{target_class}'")
            
            # STEP 2: SAM3 full image segmentation
            sam_results = segment_full_image_sam3(image_path, target_class, sam_model, sam_processor, device)
            
            if sam_results is None or len(sam_results["obbs"]) == 0:
                if image_counter % 50 == 0:
                    print("[Skip] SAM3 found no candidates")
                continue
            
            # STEP 3: Qwen selects best OBB
            pred_obb, selected_idx = select_best_obb_qwen(
                image_path, desc, sam_results["obbs"], qwen_model, qwen_processor
            )
            
            if pred_obb is None:
                if image_counter % 50 == 0:
                    print("[Skip] Qwen failed to select OBB")
                continue
            
            # Compute IoU
            iou = 0.0
            pred_obb_norm = evaluator.obb_absolute_to_normalized(pred_obb)
            gt_obb_norm = evaluator.obb_absolute_to_normalized(gt_obb)
            
            iou = evaluator.compute_iou(pred_obb_norm, gt_obb_norm)
            total_predictions += 1
            
            if is_horizontal:
                total_horizontal_gt += 1
                if iou >= 0.3:
                    correct_at_03 += 1
            else:
                total_non_horizontal_gt += 1
                if iou >= 0.5:
                    correct_at_05 += 1
            
            if image_counter % 50 == 0:
                print(f"Predicted OBB: {pred_obb}")
                print(f"Ground Truth OBB: {gt_obb}")
                print(f"IoU: {iou:.4f}")
            
            # Save result
            result_entry = {
                "image_id": image_name,
                "obj_id": obj_id,
                "description": desc,
                "target_class": target_class,
                "predicted_obb": [float(x) for x in pred_obb],
                "ground_truth_obb": [float(x) for x in gt_obb],
                "iou": float(iou),
                "is_horizontal_gt": bool(is_horizontal),
                "iou_threshold": float(iou_threshold),
                "sam_metadata": sam_results["metadata"],
                "selected_mask_id": selected_idx,
                "num_candidates": len(sam_results["obbs"])
            }
            results.append(result_entry)
            
            # Visualize
            vis_dir = pass_dir if iou > 0 else failed_dir
            vis_filename = f"{image_path.stem}_obj{obj_id}_iou{iou:.3f}.png"
            visualize_result(
                image_path, pred_obb, gt_obb, desc,
                vis_dir / vis_filename, iou, 
                sam_results["metadata"], selected_idx
            )
            
            if image_counter % 100 == 0:
                print(f"\n[Progress] Processed {image_counter} predictions...")
                if total_horizontal_gt > 0:
                    print(f"  Horizontal GT Acc@0.3: {correct_at_03/total_horizontal_gt*100:.2f}%")
                if total_non_horizontal_gt > 0:
                    print(f"  Oriented GT Acc@0.5: {correct_at_05/total_non_horizontal_gt*100:.2f}%")

    # Calculate metrics
    metrics = {
        "total_predictions": total_predictions,
        "total_horizontal_gt": total_horizontal_gt,
        "total_non_horizontal_gt": total_non_horizontal_gt,
        "accuracy_horizontal@0.3": round(correct_at_03 / total_horizontal_gt * 100, 2) if total_horizontal_gt > 0 else 0.0,
        "accuracy_oriented@0.5": round(correct_at_05 / total_non_horizontal_gt * 100, 2) if total_non_horizontal_gt > 0 else 0.0,
        "overall_accuracy": round((correct_at_03 + correct_at_05) / total_predictions * 100, 2) if total_predictions > 0 else 0.0,
        "correct_horizontal": correct_at_03,
        "correct_oriented": correct_at_05,
        "mean_iou": round(sum([r['iou'] for r in results]) / len(results), 4) if results else 0.0,
        "avg_candidates_per_image": round(sum([r['num_candidates'] for r in results]) / len(results), 2) if results else 0.0
    }

    output_json = {
        "metrics": metrics,
        "results": results
    }
    
    results_file = out_path_obj / "sam3_first_results.json"
    with open(results_file, 'w') as f:
        json.dump(output_json, f, indent=2)
    
    print("\n" + "="*60)
    print("EVALUATION COMPLETE")
    print("="*60)
    print(f"Total Predictions: {total_predictions}")
    print(f"  Horizontal GT: {total_horizontal_gt}")
    print(f"  Oriented GT: {total_non_horizontal_gt}")
    print(f"Mean IoU: {metrics['mean_iou']:.4f}")
    print(f"Avg Candidates per Image: {metrics['avg_candidates_per_image']:.2f}")
    print(f"Accuracy (Horizontal GT) @ IoU 0.3: {metrics['accuracy_horizontal@0.3']}%")
    print(f"Accuracy (Oriented GT) @ IoU 0.5: {metrics['accuracy_oriented@0.5']}%")
    print(f"Overall Accuracy: {metrics['overall_accuracy']}%")
    print(f"Results saved to: {results_file}")
    print("="*60)